# Group Lab 4: AI-Assisted Scientific Coding and Responsible Agent Use

        **Week:** Week 10

        **Lab type:** Group lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Evaluate AI-assisted coding suggestions.
- Write specific prompts.
- Refactor one small part of code.
- Verify results before accepting changes.
- Record AI assistance.

        ## Earth and environmental motivation

        AI coding agents can help with code review, refactoring, tests, and documentation, but scientific responsibility stays with the student.

        ## Dataset

        `data/processed/iowa_streamflow_daily.csv` and `scripts/messy_streamflow_analysis.py`

        ## Python concepts used

        - Code review
- Prompt design
- Refactoring
- Verification
- AI disclosure

## Group Lab 3 Debrief and Collaborative Debugging (First 10 Minutes)

Open the debrief card from Group Lab 3. Two to four students or groups will share
a solved problem, an unresolved problem with evidence, or a verification choice.
The class will investigate one open problem one check at a time.

- 0-5 min: student discussion. Compare cards in small groups and
  debug one unresolved problem together.
- 5-10 min: student reports. Two to four groups report, and the
  class records one reusable lesson.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

## Why learn AI-assisted coding now?

You have enough Python experience to evaluate generated code, and final projects are approaching.
The goal is to use agents as assistants while keeping scientific judgment, unit checks, and final responsibility in human hands.

## Weak prompts and improved prompts

Weak prompts:

- Fix my code.
- Analyze this dataset.
- Make the plot better.
- Finish this project.

Improved prompts:

- Read `scripts/messy_streamflow_analysis.py` and summarize what it does. Do not edit files yet. Identify possible bugs and propose a plan.
- Refactor the repeated monthly and annual streamflow calculations into clear functions. Keep the outputs unchanged.
- Add a small test using synthetic streamflow data to verify that annual mean flow is calculated correctly.
- Improve the hydrograph using matplotlib only. Add axis labels, units, legend, and a concise caption.
- Explain any assumptions made in the code and identify where unit conversions may be needed.

In [ ]:
from pathlib import Path
import pandas as pd

script_path = PROJECT_ROOT / "scripts" / "messy_streamflow_analysis.py"
script_text = script_path.read_text(encoding="utf-8")
print(script_text[:800])

In [ ]:
stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
monthly_manual = stream.assign(month=stream["date"].dt.to_period("M")).groupby("month")["discharge_cfs"].mean()
annual_manual = stream.assign(year=stream["date"].dt.year).groupby("year")["discharge_cfs"].mean()
print(monthly_manual.head())
print(annual_manual.head())

## Guided coding: build your reference answers first

Before asking any agent to change code, compute trusted reference numbers
with code you fully understand. After every AI-assisted edit, the refactored
script must reproduce these numbers exactly. This is the single most
effective habit for safe AI-assisted work.

In [ ]:
reference_monthly = monthly_manual.copy()
reference_annual = annual_manual.copy()
print("Reference monthly means (first 3):")
print(reference_monthly.head(3))
print()
print("Reference annual means (first 3):")
print(reference_annual.head(3))

A reference is only trustworthy if you have tested the method behind it on
data small enough to check by hand:

In [ ]:
check_data = pd.DataFrame({
    "date": pd.to_datetime(["2030-01-01", "2030-01-02", "2030-02-01"]),
    "discharge_cfs": [100.0, 300.0, 50.0],
})
check_monthly = (
    check_data.assign(month=check_data["date"].dt.to_period("M"))
    .groupby("month")["discharge_cfs"].mean()
)
assert check_monthly.loc["2030-01"] == 200.0
assert check_monthly.loc["2030-02"] == 50.0
print("Synthetic check passed: January mean is 200, February mean is 50.")

## The verification checklist

Apply this list to every AI-assisted change to the messy script:

1. **Units.** The plot title claims m3/s but the data column is cfs. Any
   refactor must fix the label or convert the values, never both halfway.
2. **Paths.** The script reads `data/processed/...` relative to the working
   directory, so it only runs from the repository root. Run it from a
   terminal at the repo root with `python scripts/messy_streamflow_analysis.py`
   and confirm the failure mode from anywhere else. Robust path handling is a
   legitimate refactor target.
3. **Missing values.** Ask what happens to the means if a day is missing.
4. **Plausibility.** Iowa River daily means run from roughly 100 to
   40,000 cfs. Any refactored output far outside that range is wrong.
5. **Reproducibility.** Run the script twice; identical numbers both times.
6. **Reference match.** Refactored monthly and annual tables must equal
   `reference_monthly` and `reference_annual`.

## Hands-on Exercise A: Ask for a plan

Write a prompt asking an AI coding agent to inspect the messy script and propose improvements without editing files.
Paste the prompt and summarize the proposed plan.

## Hands-on Exercise B: Refactor one small part

Choose monthly summary, annual summary, plot function, path handling, or missing-value handling.
Show before and after code, then explain what changed.

## Hands-on Exercise C: Verification

Run the updated script, check one calculation manually, confirm figure labels and units,
and explain whether the result makes hydrologic sense.

## Hands-on Exercise D: Final project preparation

Write one AI-agent prompt for a possible final project. Ask the agent to inspect data files,
propose a feasible plan, identify data-quality problems, suggest tests, and avoid writing the full project.

## Before you finish: verify like a reviewer

Compare the refactored script's monthly and annual outputs against your
reference tables (the pattern is shown in the instructor solution with
`np.allclose`). Confirm the hydrograph axis labels carry correct units.
Then complete the `AI_USAGE_LOG.md` entry: tools used, prompts, files
changed, what you verified, and which suggestions you rejected. Unverified
AI output submitted as finished work fails the lab rubric.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Refactor one calculation from `messy_streamflow_analysis.py` into a named function. Compute a reference result before the change, add a small test, and show that the refactored result agrees. Record one suggestion you rejected and why.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Prompt and proposed plan
- [ ] Before/after refactor example
- [ ] Verification paragraph
- [ ] Final-project preparation prompt
- [ ] AI usage log update

        ## Short reflection

        Which parts of scientific programming should never be delegated without checking?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
